# Phase 2b Stage 2: Entity聚类

目标：将每个relation的entities聚类到10-15个（适合知识图谱推荐）

策略：
- 对entities > 15的relations: BERTopic grid search找最佳参数
- 对entities ≤ 15的relations: 直接跳过
- 排序优先级：10-15范围内 > 最小垃圾桶 > 接近目标范围

In [ ]:
import sys
sys.path.insert(0, '..')

from src.clustering.entity_clusterer import EntityClusterer
from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN
from itertools import product
import json

## 1. 加载数据并统计

In [ ]:
# 初始化
clusterer = EntityClusterer(
    stage1_5_result_path='../results/entity_redistribution_stage1_5_redistributed.json'
)

# 加载Stage 1.5数据
stage1_5_data = clusterer.load_stage1_5_results()

# 统计每个relation的entity数量
print("Relation Entity Counts:")
print("="*80)

relations_to_cluster = []
relations_to_skip = []

for relation in sorted(clusterer.entities_by_relation.keys()):
    n_entities = len(clusterer.entities_by_relation[relation])
    
    if n_entities > 15:
        status = "CLUSTER"
        relations_to_cluster.append(relation)
    else:
        status = "SKIP"
        relations_to_skip.append(relation)
    
    print(f"{relation:30s}: {n_entities:3d} entities [{status}]")

print(f"\n需要聚类: {len(relations_to_cluster)} relations")
print(f"直接跳过: {len(relations_to_skip)} relations ({', '.join(relations_to_skip)})")

## 2. Grid Search函数（BERTopic）

目标：找到能产生10-15个clusters的最佳参数（适合图谱推荐）

In [ ]:
# Grid search参数空间
from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN
from itertools import product

param_grid = {
    'min_cluster_size': [2, 3, 4, 5, 6, 7, 8],
    'n_neighbors': [5, 10, 15, 20],
    'n_components': [3, 5, 7],
    'min_dist': [0.0, 0.05, 0.1, 0.15]
}

def grid_search_for_relation(relation, embeddings, entities, target_min=10, target_max=15):
    """
    对单个relation进行grid search
    
    目标：找到能产生10-15个clusters的最佳参数（适合图谱推荐）
    排序标准：
    1. 优先选择10-15个clusters的
    2. 在范围内的：最大cluster百分比越小越好（避免垃圾桶）
    3. 不在范围内的：越接近10-15越好
    """
    print(f"\n{'='*80}")
    print(f"Grid search for: {relation} ({len(entities)} entities → target: {target_min}-{target_max} clusters)")
    print(f"{'='*80}")
    
    results = []
    
    for min_cs, n_neigh, n_comp, min_d in product(
        param_grid['min_cluster_size'],
        param_grid['n_neighbors'],
        param_grid['n_components'],
        param_grid['min_dist']
    ):
        # UMAP + HDBSCAN + BERTopic
        umap_model = UMAP(
            n_components=n_comp,
            n_neighbors=min(n_neigh, len(entities) - 1),
            min_dist=min_d,
            metric='cosine',
            random_state=42
        )
        
        hdbscan_model = HDBSCAN(
            min_cluster_size=min_cs,
            metric='euclidean',
            cluster_selection_method='eom',
            prediction_data=True
        )
        
        topic_model = BERTopic(
            umap_model=umap_model,
            hdbscan_model=hdbscan_model,
            verbose=False,
            calculate_probabilities=False
        )
        
        topics, _ = topic_model.fit_transform(entities, embeddings)
        
        # 统计
        n_clusters = len(set(topics)) - (1 if -1 in topics else 0)
        n_noise = sum(1 for t in topics if t == -1)
        canonical_count = n_clusters + n_noise
        
        # 找最大cluster
        max_cluster_size = 0
        for topic_id in set(topics):
            if topic_id == -1:
                continue
            size = sum(1 for t in topics if t == topic_id)
            if size > max_cluster_size:
                max_cluster_size = size
        
        max_cluster_pct = max_cluster_size / len(entities) if len(entities) > 0 else 0
        noise_pct = n_noise / len(entities) if len(entities) > 0 else 0
        
        in_target_range = target_min <= canonical_count <= target_max
        
        results.append({
            'min_cluster_size': min_cs,
            'n_neighbors': n_neigh,
            'n_components': n_comp,
            'min_dist': min_d,
            'n_clusters': n_clusters,
            'n_noise': n_noise,
            'canonical_count': canonical_count,
            'max_cluster_size': max_cluster_size,
            'max_cluster_pct': max_cluster_pct,
            'noise_pct': noise_pct,
            'in_target_range': in_target_range
        })
    
    # 排序逻辑：
    # 1. 优先选择在10-15范围内的
    # 2. 范围内：最大cluster百分比越小越好
    # 3. 范围外：越接近10-15越好
    results_sorted = sorted(results, key=lambda x: (
        not x['in_target_range'],  # False优先（在范围内的优先）
        x['max_cluster_pct'] if x['in_target_range'] else abs(x['canonical_count'] - (target_min + target_max) / 2),
        abs(x['noise_pct'] - 0.2)
    ))
    
    # 显示Top 10结果
    print(f"\nTop 10 结果:")
    print(f"{'Rank':<5} {'mcs':<4} {'nn':<4} {'nc':<4} {'md':<6} {'→Count':<8} {'#C':<4} {'Noise':<7} {'MaxC%':<7} {'Range'}")
    print("-"*80)
    
    for i, r in enumerate(results_sorted[:10], 1):
        range_mark = "✓" if r['in_target_range'] else "✗"
        print(f"{i:<5} {r['min_cluster_size']:<4} {r['n_neighbors']:<4} {r['n_components']:<4} "
              f"{r['min_dist']:<6.2f} {r['canonical_count']:<8} {r['n_clusters']:<4} "
              f"{r['noise_pct']:<7.1%} {r['max_cluster_pct']:<7.1%} {range_mark}")
    
    # 选择最佳参数
    best = results_sorted[0]
    print(f"\n✨ 最佳配置:")
    print(f"   参数: min_cluster_size={best['min_cluster_size']}, n_neighbors={best['n_neighbors']}, "
          f"n_components={best['n_components']}, min_dist={best['min_dist']}")
    print(f"   结果: {len(entities)} → {best['canonical_count']} entities "
          f"({best['n_clusters']} clusters + {best['n_noise']} noise)")
    print(f"   质量: 最大cluster {best['max_cluster_pct']:.1%}, noise {best['noise_pct']:.1%}")
    
    if not best['in_target_range']:
        print(f"   ⚠️  警告: 未在目标范围 {target_min}-{target_max}, 当前{best['canonical_count']}")
    
    return best

print(f"总共将测试 {len(param_grid['min_cluster_size']) * len(param_grid['n_neighbors']) * len(param_grid['n_components']) * len(param_grid['min_dist'])} 个参数组合")

## 3. 处理所有relations（BERTopic grid search）

In [ ]:
all_entity_mappings = {}
best_params_per_relation = {}

for idx, relation in enumerate(sorted(clusterer.entities_by_relation.keys()), 1):
    n_entities = len(clusterer.entities_by_relation[relation])
    
    print(f"\n\n{'#'*80}")
    print(f"[{idx}/15] Processing: {relation} ({n_entities} entities)")
    print(f"{'#'*80}")
    
    # 跳过小于等于15的
    if n_entities <= 15:
        print(f"⏭️  Skipping (already ≤ 15 entities)")
        all_entity_mappings[relation] = {
            ent: ent for ent in clusterer.entities_by_relation[relation]
        }
        best_params_per_relation[relation] = None
        continue
    
    # 生成embeddings
    embeddings, entities = clusterer.embed_entities_bge(relation)
    
    # Grid search找最佳参数（目标10-15个clusters）
    best_params = grid_search_for_relation(relation, embeddings, entities, target_min=10, target_max=15)
    best_params_per_relation[relation] = best_params
    
    # 用最佳参数重新聚类
    print(f"\n应用最佳参数进行聚类...")
    topic_model, topics, probs = clusterer.cluster_with_bertopic(
        relation=relation,
        embeddings=embeddings,
        entities=entities,
        min_cluster_size=best_params['min_cluster_size'],
        n_neighbors=best_params['n_neighbors'],
        n_components=best_params['n_components'],
        min_dist=best_params['min_dist'],
        verbose=False
    )
    
    # 生成mapping
    entity_mapping = clusterer.generate_entity_mapping(relation, topics, entities)
    all_entity_mappings[relation] = entity_mapping
    
    # 简要统计
    n_after = len(set(entity_mapping.values()))
    print(f"✅ {relation}: {n_entities} → {n_after} entities")

## 4. 最终统计

In [ ]:
print(f"\n{'='*80}")
print("Final Statistics")
print(f"{'='*80}")

for relation in sorted(all_entity_mappings.keys()):
    mapping = all_entity_mappings[relation]
    n_before = len(mapping)
    n_after = len(set(mapping.values()))
    compression = n_after / n_before if n_before > 0 else 1.0
    
    # 检查是否在10-15范围内
    in_range = 10 <= n_after <= 15
    status = "✓" if in_range or n_after <= 15 else "✗"
    print(f"  {status} {relation:30s}: {n_before:3d} → {n_after:3d} ({compression:.1%})")

total_before = sum(len(m) for m in all_entity_mappings.values())
total_after = sum(len(set(m.values())) for m in all_entity_mappings.values())

print(f"\n  {'TOTAL':32s}: {total_before:3d} → {total_after:3d} ({total_after/total_before:.1%})")

# 检查是否所有需要聚类的relations都在10-15范围内
needs_clustering = [r for r, m in all_entity_mappings.items() if len(m) > 15]
in_range_count = sum(1 for r in needs_clustering if 10 <= len(set(all_entity_mappings[r].values())) <= 15)

print(f"\n在10-15范围内: {in_range_count}/{len(needs_clustering)} relations")

if in_range_count == len(needs_clustering):
    print("🎉 所有需要聚类的relations都在10-15范围内！")
else:
    out_of_range = [r for r in needs_clustering if not (10 <= len(set(all_entity_mappings[r].values())) <= 15)]
    print(f"⚠️  以下relations未在范围内: {', '.join(out_of_range)}")

## 5. 保存结果

In [ ]:
# 保存entity mappings和最佳参数
clusterer.save_results(
    all_entity_mappings=all_entity_mappings,
    output_path='../results/entity_mapping_bertopic_optimized.json',
    metadata={
        'embedding_model': 'BAAI/bge-base-en-v1.5',
        'clustering_method': 'BERTopic',
        'strategy': 'per-relation grid search',
        'target_clusters_min': 10,
        'target_clusters_max': 15,
        'canonical_selection': 'alphabetical',
        'note': 'Each relation optimized for 10-15 clusters (suitable for knowledge graph)',
        'best_params_per_relation': {
            rel: {
                'min_cluster_size': params['min_cluster_size'],
                'n_neighbors': params['n_neighbors'],
                'n_components': params['n_components'],
                'min_dist': params['min_dist'],
                'result_entities': params['canonical_count'],
                'in_target_range': params['in_target_range']
            } if params else 'skipped'
            for rel, params in best_params_per_relation.items()
        }
    }
)

print("\n✅ Phase 2b Stage 2 完成！")
print(f"结果已保存到: ../results/entity_mapping_bertopic_optimized.json")